# Continuous Brain v0.11.4 — Protected Growth

**Build: `v114-protected-integration-r2`**. Open in Colab, select a GPU runtime, then **Runtime → Run all**.
This source release reuses your existing v0.11.0 bundle and sealed retention assets. It creates a new branch from exact-expansion checkpoint `08693238…577f`.

| Stage | Planning / option visibility | Target updates |
|---|---:|---:|
| shadow | 0% | 2,000 |
| micro | 2% | 1,000 |
| low | 5% | 1,000 |
| moderate | 10% | 1,000 |

Every stage requires a passing P4b regression before proceeding. The final development test uses **41001–41100**. **51001–51100 is never evaluated by this notebook.**

Download both `continuous_brain_v0.11.4.zip` and `continuous_brain_v0.11.4.zip.sha256` alongside this notebook. The upload cell copies them into Drive. Your existing 1.1 GB bundle and retention archive are reused when present.

The notebook prints every training/evaluation command. Code and active data live on local SSD; CUDA runs the models; local files are warmed into host RAM's page cache. Every 250 updates a verified checkpoint is copied to Drive. Batch 8 and FP32 remain fixed. A100 40 GB is suitable; less GPU memory may also suffice, subject to the actual runtime.


In [ ]:
from pathlib import Path
import os
os.environ['CB_V114_STRICT_NUMERICS'] = '1'
os.environ['NVIDIA_TF32_OVERRIDE'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
threads = max(1, min(os.cpu_count() or 1, 8))
for name in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'):
    os.environ[name] = str(threads)
V110 = Path('/content/drive/MyDrive/continuous_brain_v110')
V114 = Path('/content/drive/MyDrive/continuous_brain_v114')
RELEASE = V114 / 'continuous_brain_v0.11.4.zip'
RELEASE_SHA = V114 / 'continuous_brain_v0.11.4.zip.sha256'
BUNDLE = V110 / 'continuous_brain_v110_colab_bundle.tar.gz'
WORLD = V110 / 'v110_world_retention_assets.tar.gz'
WORLD_SHA = V110 / 'v110_world_retention_assets.tar.gz.sha256'
BUNDLE_SHA = 'be3bdfe94beb0561908e9f921170f32fe0a48e700d648c30931ee7f1bd0bda3e'
EXACT_SHA = '086932388a9a410af46fea8e87cc63dd4b60161e03e247bf81fc88d28cfd577f'
WORLD_ARCHIVE_SHA = '72834fc675784d3e29a766eb5be262f5f275e0e9c454bf664825d100b96af18d'
BUILD_ID = 'v114-protected-integration-r2'
ROOT = Path('/content/cb_v114_r2')
INPUTS = ROOT / 'input'
PROJECT = ROOT / 'release' / 'continuous_brain_project'
WORK = ROOT / 'work'
print('Runtime CPU threads:', threads)


## Mount Drive and check the runtime
Allow Colab's standard Drive mount prompt. The cell below checks the available GPU and records the actual configuration. It does not rent a different runtime or change your Colab subscription.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=False)
import torch, subprocess, sys, json, hashlib, shutil, zipfile, tarfile
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU, then rerun.'
torch.set_num_threads(threads)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
V110.mkdir(parents=True, exist_ok=True)
V114.mkdir(parents=True, exist_ok=True)
ROOT.mkdir(parents=True, exist_ok=True)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU RAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Local free disk GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=False)

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def upload_required(destinations):
    print('Select:', ', '.join(destinations))
    selected = files.upload()
    for name, destination in destinations.items():
        if name not in selected:
            if not destination.exists(): raise RuntimeError('Missing selected file: ' + name)
            continue
        tmp = destination.with_name(destination.name + '.partial')
        tmp.write_bytes(selected[name])
        tmp.replace(destination)
        print('Copied:', destination)
    del selected


## Transfer the release and existing assets into Drive
If this version is already present and verifies, the upload picker is skipped. Otherwise select both release files from Windows Downloads. For a missing large bundle, uploading through Google Drive's web interface into `My Drive/continuous_brain_v110/` is convenient; the notebook can also upload it here.


In [ ]:
def release_is_current():
    if not RELEASE.exists() or not RELEASE_SHA.exists(): return False
    try:
        if file_sha(RELEASE) != RELEASE_SHA.read_text().split()[0]: return False
        with zipfile.ZipFile(RELEASE) as archive:
            source = archive.read('continuous_brain_project/continuous_brain/evidence_v114.py').decode()
            return BUILD_ID in source
    except (OSError, KeyError, ValueError, zipfile.BadZipFile): return False

if not release_is_current():
    upload_required({'continuous_brain_v0.11.4.zip':RELEASE,
                     'continuous_brain_v0.11.4.zip.sha256':RELEASE_SHA})
assert release_is_current(), 'Release checksum/build mismatch. Select this release and its matching checksum.'
if not BUNDLE.exists():
    upload_required({BUNDLE.name:BUNDLE})
assert file_sha(BUNDLE) == BUNDLE_SHA, 'v0.11.0 bundle hash mismatch'
if not WORLD.exists() or not WORLD_SHA.exists():
    upload_required({WORLD.name:WORLD, WORLD_SHA.name:WORLD_SHA})
assert file_sha(WORLD) == WORLD_SHA.read_text().split()[0] == WORLD_ARCHIVE_SHA, 'Retention archive hash mismatch'
print('Release, exact-input bundle, and world archive verified.')


## Stage and verify the source
The archive extraction checks member paths. The per-file manifest must match before importing the project. Active work and existing checkpoints are retained when this cell is rerun.


In [ ]:
def extract_checked(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    def validate(name):
        path = (destination / name).resolve()
        if path != destination and destination not in path.parents:
            raise ValueError('Archive path escapes staging directory: ' + name)
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                validate(member.filename)
                if ((member.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError('Archive symlinks are not supported')
            archive.extractall(destination)
    else:
        with tarfile.open(archive_path) as archive:
            members = archive.getmembers()
            for member in members:
                validate(member.name)
                if not (member.isfile() or member.isdir()): raise ValueError('Archive links are not supported')
            archive.extractall(destination, members=members)

# Source is extracted into a directory identified by the release hash.
source_root = ROOT / ('release_' + file_sha(RELEASE)[:16])
extract_checked(RELEASE, source_root)
PROJECT = source_root / 'continuous_brain_project'
manifest = json.loads((PROJECT / 'MANIFEST_SHA256.json').read_text())
bad = [name for name, digest in manifest.items()
       if not (PROJECT/name).is_file() or file_sha(PROJECT/name) != digest]
assert not bad, 'Source manifest mismatch: ' + repr(bad[:10])
marker = INPUTS / 'VERIFIED_BUNDLE_SHA256.txt'
if not marker.exists() or marker.read_text().strip() != BUNDLE_SHA:
    if shutil.disk_usage('/content').free < 10 * 1024**3:
        raise RuntimeError('At least 10 GiB of local staging space is required')
    extract_checked(BUNDLE, INPUTS)
    marker.write_text(BUNDLE_SHA)
exact_checkpoint = INPUTS/'payload/state/child_v110_growth/infant_training.pt'
assert file_sha(exact_checkpoint) == EXACT_SHA
subprocess.run([sys.executable,'-m','pip','install','-e',str(PROJECT),'--no-deps','--no-build-isolation'],check=True)
# Avoid accidentally continuing with a previously imported version in this kernel.
loaded = sys.modules.get('continuous_brain')
if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise RuntimeError('Another release was already imported. Restart the Colab session and Run all.')
sys.path.insert(0, str(PROJECT))
import continuous_brain
from continuous_brain.evidence_v114 import BUILD_ID as INSTALLED_BUILD
assert continuous_brain.__version__ == '0.11.4' and INSTALLED_BUILD == BUILD_ID
print('Verified source files:', len(manifest))


## Create or resume the protected branch
This build uses `state/child_v114_protected_r2` and `results/protected_integration_r2` in your v0.11.4 Drive directory. It requires the exact source hash and verifies the frozen legacy tensor fingerprint on resume. Older experimental v0.11.4 branches stay available.


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('cb114_colab_runner', PROJECT/'scripts/run_v114_colab.py')
runner_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner_module)
run = runner_module.ColabRun(PROJECT, INPUTS, WORK, V114, WORLD)
run.stage_inputs()
run.warm_cache()


## Baseline and protected stages
Baseline: exact expansion on P4b seeds **12001–12100**, 100 episodes, 120 steps. Each stage then uses the same evaluation protocol. The runner reuses a result only when its checkpoint hash, runtime, source manifest, and command settings match.

Training uses the old Detour nursery for compatibility anchors and successful option rehearsal, and the randomized training nursery for new learning. Each stage has a cumulative target; reruns train only remaining updates. A failed P4b result stops the loop, preserving the endpoint and logs.


In [ ]:
baseline = run.baseline()
stage_results = run.run_stages()


## Final retention and development qualification
Run bootstrap retention, verify the original world seal and transition digests, then evaluate the relocated Foraging/Detour holdouts. The original seal is preserved; only its two paths change in a separate relocation copy.

Randomized-detour development runs after retention passes. Completion must reach **20%**. No reserved blind range is evaluated.


In [ ]:
bootstrap, world, generalized = run.qualify()
print('Bootstrap retention:', bootstrap['passed'])
print('World retention:', world['world_consolidation_passed'])
print('Randomized development completion:', generalized['completion_rate'])


## Final measured report
A readiness failure is saved as a failure. Missing results and old-checkpoint results cannot be filled in as passes. The final report is a development gate; the new release has not demonstrated a planning recovery until your checkpoint completes these evaluations.


In [ ]:
readiness = run.readiness()
print(json.dumps(readiness, indent=2))
print('Final report:', run.durable_results/'V114_FINAL_SUMMARY.json')
print('Readiness report:', run.durable_results/'V114_PROTECTED_READINESS.json')


## If a cell stops

Read the error and the corresponding log in `MyDrive/continuous_brain_v114/results/protected_integration_r2/`. For a Colab disconnect, reconnect a GPU runtime and use **Run all**; the last verified periodic checkpoint is resumed. Up to 249 updates since the last durable save can require repeating.

If P4b or retention fails, send the saved JSON and log for analysis. Keep the stage endpoint; do not increase its target updates or open the next gate manually. If the runtime changes, cached qualification results are re-evaluated against their frozen endpoints on the new runtime.

The original exact expansion and failed Phase A checkpoints remain useful experimental records. This notebook trains only the new protected branch.
